In [1]:
from pathlib import Path
import os

PROJECT_NAME = "MALDIAlign"

cwd = Path().resolve()

# Walk upwards until we find the project folder
target = None
for parent in [cwd] + list(cwd.parents):
    if parent.name == PROJECT_NAME:
        target = parent
        break

# If the project folder is found and we are not already there, then change cwd
if target is not None and target != cwd:
    os.chdir(target)

print("Working directory:", os.getcwd())

Working directory: /export/usuarios01/agnavarr/MALDIAlign


In [2]:
import numpy as np
import pandas as pd

from utils.load_config import load_config
from utils.load_data import load_pkl

## Data loading

In [3]:
cfg = load_config()
driams_pkl = cfg["data"]["DRIAMS_REDUCED_PKL"]

In [4]:
driams = load_pkl(driams_pkl)

In [5]:
data, label, meta = driams["data"], driams["label"], driams["meta"]

In [6]:
meta = pd.DataFrame.from_records(list(meta))

In [7]:
type(data), type(label), type(meta)

(numpy.ndarray, numpy.ndarray, pandas.core.frame.DataFrame)

In [8]:
np.unique(label)

array(['Enterobacter_cloacae_complex', 'Enterococcus_Faecium',
       'Escherichia_Coli', 'Klebsiella_Pneumoniae',
       'Pseudomonas_Aeruginosa', 'Staphylococcus_Aureus'], dtype='<U28')

In [9]:
species, counts = np.unique(label, return_counts=True)
for sp, n in zip(species, counts):
    print(f"{sp}: {n}")

Enterobacter_cloacae_complex: 3351
Enterococcus_Faecium: 2136
Escherichia_Coli: 11160
Klebsiella_Pneumoniae: 6791
Pseudomonas_Aeruginosa: 5824
Staphylococcus_Aureus: 10519


In [10]:
meta["hospital"].value_counts()

hospital
DRIAMS_A    27810
DRIAMS_D     7308
DRIAMS_C     2677
DRIAMS_B     1986
Name: count, dtype: int64

In [15]:
meta

,hospital,year,genus,species,study
0,DRIAMS_B,2018,Klebsiella,Pneumoniae,23a9e0ba-d4b2-40eb-808c-37270a5117dc.txt
1,DRIAMS_B,2018,Klebsiella,Pneumoniae,36437826-52fa-4daf-8f2b-4133f4307156.txt
2,DRIAMS_B,2018,Klebsiella,Pneumoniae,da983ae0-e887-4771-bdb1-9a8d564eb017.txt
3,DRIAMS_B,2018,Klebsiella,Pneumoniae,5e8f4603-1399-4ad5-a618-7e8bf2fc4207.txt
4,DRIAMS_B,2018,Klebsiella,Pneumoniae,ce63ff87-717d-4eac-8dca-590ffa840867.txt
...,...,...,...,...,...
39776,DRIAMS_A,2017,Enterobacter,Ludwigii,12a9f8dc-2ea4-4e61-9cb3-8a318e207b72_MALDI1.txt
39777,DRIAMS_C,2018,Enterobacter,Ludwigii,28082703960_A6.txt
39778,DRIAMS_C,2018,Enterobacter,Ludwigii,28020109560_F1.txt
39779,DRIAMS_C,2018,Enterobacter,Ludwigii,28030306620_F3.txt


In [11]:
# Filter the data by hospital
filtered_data = {}
for hosp in meta["hospital"].unique():
    idx = np.where(meta["hospital"].values == hosp)[0]
    filtered_data[hosp] = {
        "data": data[idx],
        "label": label[idx],
        "meta": meta.iloc[idx]
    }

In [12]:
species, counts = np.unique(filtered_data["DRIAMS_A"]["label"], return_counts=True)
for sp, n in zip(species, counts):
    print(f"{sp}: {n}")

Enterobacter_cloacae_complex: 2573
Enterococcus_Faecium: 1800
Escherichia_Coli: 7381
Klebsiella_Pneumoniae: 4083
Pseudomonas_Aeruginosa: 4908
Staphylococcus_Aureus: 7065


In [14]:
# Declare the datasets
dataA, labelA, metaA = filtered_data["DRIAMS_A"]["data"], filtered_data["DRIAMS_A"]["label"], filtered_data["DRIAMS_A"]["meta"]
dataB, labelB, metaB = filtered_data["DRIAMS_B"]["data"], filtered_data["DRIAMS_B"]["label"], filtered_data["DRIAMS_B"]["meta"]
dataC, labelC, metaC = filtered_data["DRIAMS_C"]["data"], filtered_data["DRIAMS_C"]["label"], filtered_data["DRIAMS_C"]["meta"]
dataD, labelD, metaD = filtered_data["DRIAMS_D"]["data"], filtered_data["DRIAMS_D"]["label"], filtered_data["DRIAMS_D"]["meta"]

## Models

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(dataA, labelA, test_size=0.2, shuffle=True, stratify=labelA)

### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Define the pipeline
pipe_logistic = Pipeline([
    ('scaler', 'passthrough'),
    ('lr', LogisticRegression(solver='liblinear'))
])

# Define the hyperparameter grid
params = [
    {
        'scaler': ['passthrough', StandardScaler()],
        'lr__C': np.logspace(-3, 3, 10),
        'lr__penalty': ['l1', 'l2'],
    }
]

# Define and the GridSearchCV 
grid_logistic = GridSearchCV(estimator=pipe_logistic, param_grid=params, cv=5, scoring='balanced_accuracy', n_jobs=-1, verbose=1)
grid_logistic.fit(X_train, y_train)

Fitting 5 folds for each of 40 candidates, totalling 200 fits


/usr/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/usr/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/usr/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the est

In [ ]:
# Obtain the best parameters
print("Best parameters:", grid_logisitc,best_params_)
print("Best balanced accuracy:", grid_logistic.best_score_)

In [ ]:
# Train

### SVM-Linear

### SVM-RBF

### LightGBM

### RF